# COMP6009 Assignment Part A - Notebook 00: setup
Group 01

In this notebook we set up everything the other notebooks use:
- mount Google Drive and make our folders
- write our code (`cointex_parta`) to Drive
- clone CoinTex and check our level table against `cointex.kv`
- run some quick tests and save the experiment settings

Folder layout on Drive:
```
COMP6009_Assignment-Part A/
  code/cointex_parta/     our code (simulator + greedy, GA and our agent)
  config/                 experiment settings
  data/                   cointex.kv, level table
  results/01_greedy_baseline, 02_proposed, 03_ga_task2, 04_comparison
  report_assets/          figures and tables for the report and slides
  deliverables/           results summary
  logs/                   run log
```

In [1]:
# mount google drive and make the folders we save everything into
import os, sys, json, subprocess, datetime, importlib
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/COMP6009_Assignment-Part A'   # all our files go here
PATHS = {
    'code':         f'{ROOT}/code',                          # the cointex_parta package
    'data':         f'{ROOT}/data',                          # cointex.kv, level table
    'config':       f'{ROOT}/config',                        # experiment_config.json
    'greedy':       f'{ROOT}/results/01_greedy_baseline',
    'proposed':     f'{ROOT}/results/02_proposed',
    'ga':           f'{ROOT}/results/03_ga_task2',
    'compare':      f'{ROOT}/results/04_comparison',
    'report_figs':  f'{ROOT}/report_assets/figures',         # final figures for the report/slides
    'report_tabs':  f'{ROOT}/report_assets/tables',          # final tables for the report
    'deliverables': f'{ROOT}/deliverables',
    'logs':         f'{ROOT}/logs',
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pygad==3.3.1'], check=True)
if PATHS['code'] not in sys.path:
    sys.path.insert(0, PATHS['code'])

def log_run(notebook, **info):
    # keep a log of every run
    info.update(notebook=notebook, at=datetime.datetime.now().isoformat(timespec='seconds'))
    with open(f"{PATHS['logs']}/run_log.jsonl", 'a') as f:
        f.write(json.dumps(info) + '\n')

print('Project root:', ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/COMP6009_Assignment-Part A


## 1. Write our code to Drive
Each cell writes one file of our package. The other notebooks import it from here.

In [2]:
PKG = f"{PATHS['code']}/cointex_parta"
os.makedirs(f'{PKG}/agents', exist_ok=True)
print(PKG)

/content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta


In [3]:
%%writefile "{PKG}/__init__.py"
"""COMP6009 Assignment Part A - CoinTex search agents (Group 01)."""
__version__ = "1.0.0"

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/__init__.py


In [4]:
%%writefile "{PKG}/levels.py"
"""
Settings for all 24 levels.

- num_coins and num_collisions_level: from the LevelN classes in main_work_greedy.py
- monster speeds and fire paths: from the <LevelN> parts of cointex.kv

Note: level 20 has 10 fires in the kv file but num_fires = 8, and level 22 has
4 monsters but num_monsters = 2, so only the first ones move in the game.
We do the same here.

parse_level_table() reads the table back from the game files so we can check it
(notebook 00).
"""

import re

NUM_LEVELS = 24

# monsters: list of (anim_duration_low, anim_duration_high)
# fires:    list of ((start_x, start_y), (end_x, end_y), leg_duration) in pos_hint units
LEVEL_TABLE = {
    1: dict(num_coins=5, num_collisions_level=20,
         monsters=[(3.5, 4.5)],
         fires=[]),
    2: dict(num_coins=8, num_collisions_level=30,
         monsters=[(2.5, 3.5)],
         fires=[]),
    3: dict(num_coins=12, num_collisions_level=30,
         monsters=[(1.0, 1.5)],
         fires=[]),
    4: dict(num_coins=10, num_collisions_level=20,
         monsters=[(2.5, 3.5)],
         fires=[((1.5, 0.5), (-0.5, 0.5), 2.5)]),
    5: dict(num_coins=15, num_collisions_level=20,
         monsters=[(1.0, 1.5)],
         fires=[((1.5, 0.7), (-0.5, 0.7), 3.0), ((1.5, 0.3), (-0.5, 0.3), 2.0)]),
    6: dict(num_coins=12, num_collisions_level=20,
         monsters=[(2.5, 3.0)],
         fires=[((1.5, 0.8), (-0.5, 0.8), 2.5), ((1.5, 0.2), (-0.5, 0.2), 2.0), ((0.5, 1.5), (0.5, -0.5), 3.5)]),
    7: dict(num_coins=10, num_collisions_level=25,
         monsters=[(1.0, 1.5), (1.5, 2.5), (1.5, 2.5)],
         fires=[]),
    8: dict(num_coins=15, num_collisions_level=25,
         monsters=[(1.5, 2.5), (1.0, 1.5)],
         fires=[]),
    9: dict(num_coins=12, num_collisions_level=25,
         monsters=[(2.5, 3.5), (1.2, 1.7)],
         fires=[]),
    10: dict(num_coins=14, num_collisions_level=30,
         monsters=[(1.5, 2.5), (0.8, 1.2), (1.0, 1.5)],
         fires=[]),
    11: dict(num_coins=15, num_collisions_level=30,
         monsters=[(2.5, 3.5), (2.0, 2.5)],
         fires=[((1.5, 0.5), (-0.5, 0.5), 3.5)]),
    12: dict(num_coins=12, num_collisions_level=30,
         monsters=[(2.5, 3.5), (1.5, 2.0)],
         fires=[((1.5, 0.5), (-0.5, 0.5), 2.5)]),
    13: dict(num_coins=10, num_collisions_level=20,
         monsters=[(2.5, 3.5), (2.5, 3.0)],
         fires=[((1.5, 0.5), (-0.5, 0.5), 2.5), ((0.5, 1.5), (0.5, -0.5), 3.5)]),
    14: dict(num_coins=15, num_collisions_level=30,
         monsters=[],
         fires=[((1.5, 0.2), (-0.5, 0.2), 2.5), ((1.5, 0.8), (-0.5, 0.8), 3.5), ((-1.5, 0.5), (-0.5, 0.5), 3.5), ((0.8, 1.5), (0.8, -0.5), 2.5), ((0.2, 1.5), (0.2, -0.5), 3.5), ((0.5, 1.5), (0.5, -0.5), 2.5)]),
    15: dict(num_coins=16, num_collisions_level=30,
         monsters=[(3.0, 3.5), (3.0, 3.5)],
         fires=[((1.2, 0.2), (-0.5, 0.2), 2.5), ((1.5, 0.7), (-0.5, 0.7), 3.5), ((0.5, 1.5), (0.5, -0.5), 2.5)]),
    16: dict(num_coins=15, num_collisions_level=35,
         monsters=[(2.0, 3.5), (3.0, 3.5), (1.0, 1.5)],
         fires=[((1.5, 0.2), (-0.5, 0.2), 2.5), ((1.5, 0.7), (-0.5, 0.7), 1.5)]),
    17: dict(num_coins=10, num_collisions_level=30,
         monsters=[],
         fires=[((1.0, 0.3), (0.0, 0.3), 2.5), ((1.0, 0.8), (0.0, 0.8), 2.0), ((0.8, 1.0), (0.8, 0.0), 3.5), ((0.3, 0.0), (0.3, 1.0), 3.0)]),
    18: dict(num_coins=15, num_collisions_level=30,
         monsters=[(1.0, 1.6), (1.0, 2.0), (1.0, 1.2)],
         fires=[((0.8, 0.3), (0.2, 0.3), 3.2), ((0.3, 0.8), (0.7, 0.8), 2.3), ((0.8, 0.3), (0.8, 0.7), 3.2), ((0.3, 0.8), (0.3, 0.2), 2.5)]),
    19: dict(num_coins=12, num_collisions_level=30,
         monsters=[],
         fires=[((0.8, 0.3), (0.2, 0.3), 2.2), ((0.3, 0.8), (0.7, 0.8), 3.0), ((-1.5, 0.5), (1.5, 0.5), 2.5), ((0.8, 0.3), (0.8, 0.7), 3.2), ((0.3, 0.8), (0.3, 0.2), 2.3), ((0.5, -1.5), (0.5, 1.5), 2.4)]),
    20: dict(num_coins=15, num_collisions_level=30,
         monsters=[],
         fires=[((0.9, 0.1), (0.1, 0.1), 2.5), ((0.0, 0.3), (1.0, 0.3), 2.6), ((0.1, 0.5), (0.9, 0.5), 2.5), ((0.1, 0.9), (0.9, 0.9), 3.3), ((0.2, 0.1), (0.2, 0.9), 2.5), ((0.5, 0.9), (0.5, 0.1), 3.3), ((0.7, -1.5), (0.7, 1.5), 2.6), ((0.9, 0.1), (0.9, 0.9), 3.3)]),
    21: dict(num_coins=18, num_collisions_level=30,
         monsters=[(1.0, 1.3), (1.0, 1.5)],
         fires=[((0.8, 0.3), (0.2, 0.3), 1.8), ((0.3, 0.8), (0.7, 0.8), 2.5), ((-1.5, 0.5), (1.5, 0.5), 3.0), ((0.1, 0.3), (0.9, 0.3), 2.5)]),
    22: dict(num_coins=20, num_collisions_level=30,
         monsters=[(1.0, 1.8), (1.0, 2.0)],
         fires=[((0.8, 0.3), (0.2, 0.3), 2.9), ((0.3, 0.8), (0.7, 0.8), 2.5), ((-1.5, 0.5), (1.5, 0.5), 2.3), ((0.1, 0.3), (0.9, 0.3), 2.8)]),
    23: dict(num_coins=25, num_collisions_level=30,
         monsters=[(1.5, 2.0), (1.5, 2.5)],
         fires=[((-1.5, 0.3), (1.0, 0.3), 2.8), ((1.5, 0.8), (-0.5, 0.8), 2.0)]),
    24: dict(num_coins=20, num_collisions_level=30,
         monsters=[(1.5, 1.7), (1.8, 2.5), (1.5, 2.0)],
         fires=[((0.8, 0.3), (0.2, 0.3), 2.8), ((0.3, 0.8), (0.7, 0.8), 2.5)]),
}


def level_summary(level):
    cfg = LEVEL_TABLE[level]
    return {"level": level, "coins": cfg["num_coins"],
            "monsters": len(cfg["monsters"]), "fires": len(cfg["fires"]),
            "life_frames": cfg["num_collisions_level"]}


def parse_level_table(kv_text, game_py_text):
    # read the level settings from cointex.kv and the game .py file
    levels = {}
    blocks = re.split(r"\n<Level(\d+)>:", kv_text)
    for i in range(1, len(blocks), 2):
        n = int(blocks[i])
        body = blocks[i + 1].split("\n<")[0]
        mons = [(int(k), float(a), float(b)) for k, a, b in re.findall(
            r"id:\s*monster(\d+)_image_lvl\d+\s*\n\s*monst_anim_duration_low:\s*([\d.]+)"
            r"\s*\n\s*monst_anim_duration_high:\s*([\d.]+)", body)]
        fires = [(int(k), (float(sx), float(sy)), (float(ex), float(ey)), float(d))
                 for k, sx, sy, ex, ey, d in re.findall(
            r"id:\s*fire(\d+)_lvl\d+\s*\n\s*fire_start_pos_hint:\s*\{'x':\s*([-\d.]+),\s*'y':\s*([-\d.]+)\}"
            r"\s*\n\s*fire_end_pos_hint:\s*\{'x':\s*([-\d.]+),\s*'y':\s*([-\d.]+)\}"
            r"\s*\n\s*fire_anim_duration:\s*([\d.]+)", body)]
        levels[n] = {"monsters": mons, "fires": fires}

    for m in re.finditer(r"class Level(\d+)\(.*?\):(.*?)(?=\nclass |\napp = )", game_py_text, re.S):
        n, body = int(m.group(1)), m.group(2)

        def grab(key):
            return float(re.search(key + r"\s*=\s*([\d.]+)", body).group(1))
        levels[n]["num_coins"] = int(grab("num_coins"))
        levels[n]["num_collisions_level"] = int(grab("num_collisions_level"))
        n_mon, n_fire = int(grab("num_monsters")), int(grab("num_fires"))
        levels[n]["monsters"] = [x[1:] for x in sorted(levels[n]["monsters"]) if x[0] <= n_mon]
        levels[n]["fires"] = [x[1:] for x in sorted(levels[n]["fires"]) if x[0] <= n_fire]
    return levels


def diff_against(parsed):
    # list of differences between LEVEL_TABLE and the parsed table (empty = same)
    problems = []
    for n in range(1, NUM_LEVELS + 1):
        a, b = LEVEL_TABLE[n], parsed.get(n)
        if b is None:
            problems.append(f"level {n} missing from parsed table")
            continue
        for key in ("num_coins", "num_collisions_level", "monsters", "fires"):
            if a[key] != b[key]:
                problems.append(f"level {n} {key}: embedded={a[key]} parsed={b[key]}")
    return problems


def verify_kv(kv_text):
    # check the monster/fire settings against cointex.kv only.
    # we can't use the repo's main.py for the coin counts because it is different
    # from main_work_greedy.py (levels 1, 17, 19, 20 have other coin counts)
    stub = "\n".join(
        f"class Level{n}(Screen):\n    num_coins = {c['num_coins']}\n"
        f"    num_collisions_level = {c['num_collisions_level']}\n"
        f"    num_monsters = {len(c['monsters'])}\n    num_fires = {len(c['fires'])}\n"
        for n, c in LEVEL_TABLE.items()) + "\napp = None\n"
    return diff_against(parse_level_table(kv_text, stub))

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/levels.py


In [5]:
%%writefile "{PKG}/sim.py"
"""
Headless version of the CoinTex game so we can run lots of episodes without
Kivy (Kivy needs a window, so it won't run on Colab).

We copied the numbers from the game files:
- sprite sizes from cointex.kv (character/monster 0.15, fire 0.03, coin 0.05)
- collision checks from main_work_greedy.py (coin and fire use width/3,
  monster uses width/2), counted every frame at 60 fps
- the character always takes 0.25 s to walk to its target
- monsters walk to random points, fires sweep back and forth on fixed paths

Positions are sprite centres (the game uses pos_hint = bottom-left corner).
Same seed = same coins and same monster moves, so every agent gets the
exact same world.
"""

import collections
import random

import numpy as np

from .levels import LEVEL_TABLE

FPS = 60
DT = 1.0 / FPS
MOVE_SECONDS = 0.25          # char_anim_duration, same for all levels
MOVE_FRAMES = int(round(MOVE_SECONDS * FPS))

CHAR_SIZE = 0.15
MONSTER_SIZE = 0.15
FIRE_SIZE = 0.03
COIN_SIZE = 0.05

CHAR_HALF = CHAR_SIZE / 2.0
MONSTER_GAB = CHAR_SIZE / 2.0    # width/2 in monst_pos_hint
COIN_GAB = CHAR_SIZE / 3.0       # width/3 in char_pos_hint
FIRE_GAB = CHAR_SIZE / 3.0       # width/3 in fire_pos_hint

# coins can be right at the edge, so the character has to be allowed there too
EDGE_MARGIN = COIN_SIZE / 2.0

FIRE_PARK = (1.1, 1.1)       # where screen_on_pre_enter puts the fires
MONSTER_PARK = (0.8, 0.8)    # monster start position in the kv file

TIME_LIMIT_SECONDS = 60.0

# for our near-miss metric: close to a hazard but not touching it
NEAR_MISS_BAND = 0.05

FireState = collections.namedtuple(
    "FireState", "x y vx vy tx ty remaining ox oy leg_duration")
Observation = collections.namedtuple(
    "Observation",
    "level frame time char coins monsters fires margin stagnation life")


class CoinTexSim:
    """One level of the game."""

    def __init__(self, level_num, seed, time_limit=TIME_LIMIT_SECONDS, hazard_speed=1.0):
        # hazard_speed > 1 speeds up monsters and fires (only used for the stress test)
        cfg = LEVEL_TABLE[level_num]
        self.hazard_speed = float(hazard_speed)
        self.level_num = level_num
        self.seed = seed
        self.num_coins = cfg["num_coins"]
        self.num_collisions_level = cfg["num_collisions_level"]
        self.time_limit = time_limit
        self.rng = random.Random(seed)

        self.frames = 0
        self.collisions_hit = 0
        self.monster_hit_frames = 0
        self.fire_hit_frames = 0
        self.near_miss_frames = 0
        self.path_length = 0.0
        self.killed = False
        self.frames_since_coin = 0
        self.coin_times = []

        # character starts at pos_hint (0, 0)
        self.agent = np.array([CHAR_HALF, CHAR_HALF])
        self._anim = None
        self._build_layout(cfg)

    def _build_layout(self, cfg):
        # coins: one per vertical strip, same as screen_on_pre_enter
        self.coins = []
        section_width = 1.0 / self.num_coins
        for k in range(self.num_coins):
            x = self.rng.uniform(section_width * k, section_width * (k + 1) - COIN_SIZE)
            y = self.rng.uniform(0.0, 1.0 - COIN_SIZE)
            self.coins.append(np.array([x + COIN_SIZE / 2.0, y + COIN_SIZE / 2.0]))

        self.monsters = []
        for low, high in cfg["monsters"]:
            low, high = low / self.hazard_speed, high / self.hazard_speed
            start = np.array([MONSTER_PARK[0] + MONSTER_SIZE / 2.0,
                              MONSTER_PARK[1] + MONSTER_SIZE / 2.0])
            monster = {"pos": start, "prev": start.copy(), "low": low, "high": high}
            self._retarget_monster(monster)
            self.monsters.append(monster)

        half = FIRE_SIZE / 2.0
        self.fires = []
        for start, end, duration in cfg["fires"]:
            duration = duration / self.hazard_speed
            p0 = np.array([start[0] + half, start[1] + half])
            p1 = np.array([end[0] + half, end[1] + half])
            park = np.array([FIRE_PARK[0] + half, FIRE_PARK[1] + half])
            # first the fire moves from the park spot to its start point
            self.fires.append({"p0": p0, "p1": p1, "duration": duration,
                               "pos": park.copy(), "prev": park.copy(),
                               "leg_from": park.copy(), "leg_to": p0.copy(),
                               "next": p1.copy(), "elapsed": 0.0})

    def _retarget_monster(self, monster):
        # same as monst_animation_completed in the game
        pos_hint = (self.rng.uniform(0.0, 1 - MONSTER_SIZE / 4),
                    self.rng.uniform(0.0, 1 - MONSTER_SIZE / 4))
        monster["target"] = np.array([pos_hint[0] + MONSTER_SIZE / 2.0,
                                      pos_hint[1] + MONSTER_SIZE / 2.0])
        monster["duration"] = self.rng.uniform(monster["low"], monster["high"])
        monster["elapsed"] = 0.0
        monster["origin"] = monster["pos"].copy()

    def command(self, target):
        # start walking to target (cancels the current walk)
        if target is None:
            return
        self._anim = {"start": self.agent.copy(),
                      "target": np.array(target, dtype=float),
                      "elapsed": 0.0}

    def _advance_character(self):
        if self._anim is None:
            return
        self._anim["elapsed"] += DT
        fraction = min(1.0, self._anim["elapsed"] / MOVE_SECONDS)
        new = self._anim["start"] + (self._anim["target"] - self._anim["start"]) * fraction
        self.path_length += float(np.hypot(*(new - self.agent)))
        self.agent = new
        if fraction >= 1.0:
            self._anim = None

    def _advance_monsters(self):
        for monster in self.monsters:
            monster["prev"] = monster["pos"].copy()
            monster["elapsed"] += DT
            fraction = min(1.0, monster["elapsed"] / monster["duration"])
            monster["pos"] = monster["origin"] + (monster["target"] - monster["origin"]) * fraction
            if fraction >= 1.0 - 1e-9:
                self._retarget_monster(monster)

    def _advance_fires(self):
        for fire in self.fires:
            fire["prev"] = fire["pos"].copy()
            fire["elapsed"] += DT
            if fire["elapsed"] >= fire["duration"] - 1e-9:
                # end of a sweep, turn around. keep the leftover time so the
                # timing doesn't drift (we found this when testing the predictor)
                fire["elapsed"] = max(0.0, fire["elapsed"] - fire["duration"])
                fire["leg_from"], fire["leg_to"], fire["next"] = (
                    fire["leg_to"].copy(), fire["next"].copy(), fire["leg_to"].copy())
            fraction = min(1.0, fire["elapsed"] / fire["duration"])
            fire["pos"] = fire["leg_from"] + (fire["leg_to"] - fire["leg_from"]) * fraction

    def _collect_coins(self):
        before = len(self.coins)
        self.coins = [c for c in self.coins
                      if not (abs(self.agent[0] - c[0]) <= COIN_GAB
                              and abs(self.agent[1] - c[1]) <= COIN_GAB)]
        collected = before - len(self.coins)
        if collected:
            self.coin_times.extend([self.frames * DT] * collected)
            self.frames_since_coin = 0
        else:
            self.frames_since_coin += 1

    def _check_hazards(self):
        # one hit per frame of contact, like the game
        near = False
        for monster in self.monsters:
            d = np.abs(self.agent - monster["pos"])
            if d[0] <= MONSTER_GAB and d[1] <= MONSTER_GAB:
                self.collisions_hit += 1
                self.monster_hit_frames += 1
            elif max(d) <= MONSTER_GAB + NEAR_MISS_BAND:
                near = True
        for fire in self.fires:
            d = np.abs(self.agent - fire["pos"])
            if d[0] <= FIRE_GAB and d[1] <= FIRE_GAB:
                self.collisions_hit += 1
                self.fire_hit_frames += 1
            elif max(d) <= FIRE_GAB + NEAR_MISS_BAND:
                near = True
        if near:
            self.near_miss_frames += 1
        if self.collisions_hit >= self.num_collisions_level:
            self.killed = True

    def step_frame(self):
        self.frames += 1
        self._advance_character()
        self._advance_monsters()
        self._advance_fires()
        self._collect_coins()
        self._check_hazards()

    def run_frames(self, n):
        for _ in range(int(n)):
            if self.done:
                return
            self.step_frame()

    def observe(self):
        # what the agent can see. velocity = change since last frame, which is
        # something you could also measure in the real game
        monsters = tuple(
            (float(m["pos"][0]), float(m["pos"][1]),
             float((m["pos"][0] - m["prev"][0]) * FPS), float((m["pos"][1] - m["prev"][1]) * FPS))
            for m in self.monsters)
        fires = []
        for f in self.fires:
            remaining = max(f["duration"] - f["elapsed"], 0.0)
            fires.append(FireState(
                float(f["pos"][0]), float(f["pos"][1]),
                float((f["pos"][0] - f["prev"][0]) * FPS), float((f["pos"][1] - f["prev"][1]) * FPS),
                float(f["leg_to"][0]), float(f["leg_to"][1]), remaining,
                float(f["next"][0]), float(f["next"][1]), f["duration"]))
        return Observation(
            level=self.level_num, frame=self.frames, time=self.frames * DT,
            char=(float(self.agent[0]), float(self.agent[1])),
            coins=tuple((float(c[0]), float(c[1])) for c in self.coins),
            monsters=monsters, fires=tuple(fires),
            margin=EDGE_MARGIN, stagnation=self.frames_since_coin * DT,
            life=self.remaining_life_fraction())

    @property
    def seconds(self):
        return self.frames * DT

    @property
    def won(self):
        return len(self.coins) == 0

    @property
    def timed_out(self):
        return not self.won and not self.killed and self.seconds >= self.time_limit

    @property
    def done(self):
        return self.won or self.killed or self.seconds >= self.time_limit

    def remaining_life_fraction(self):
        return max(0.0, 1.0 - self.collisions_hit / self.num_collisions_level)

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/sim.py


In [6]:
%%writefile "{PKG}/agents/__init__.py"
"""Agents: greedy (original baseline), ga_task2 (Lab Task 2 GA), proposed (RH-STA*)."""

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/agents/__init__.py


In [7]:
%%writefile "{PKG}/agents/greedy.py"
"""
The original greedy agent from main_work_greedy.py.

create_matrix, check_coin and findMinRoute are copied from the given file
(we only took out the print statements), so the baseline behaves the same.
GreedyAgent below just replays what CollectCoinThread.run() does:
plan a nearest-neighbour route over the coins, walk to each coin with a
0.4 s sleep, then read the positions again and repeat.

Problems we found (see notebook 01):
  1. hazards are only checked when the route is planned, not while walking it
  2. check_coin pops from the list it is looping over, so it skips coins
  3. the route always starts from (0.1, 0.1), not the real position
  4. moves have no length limit, so one move can cross a monster's path
  5. 0.4 s sleep but the walk only takes 0.25 s
  6. if all coins are near a hazard the route is empty and it just waits
"""

from typing import DefaultDict

from ..sim import COIN_SIZE, FIRE_SIZE, MONSTER_SIZE, FPS

INT_MAX = 2147483647
SLEEP_SECONDS = 0.4


# ---- copied from main_work_greedy.py ----
def create_matrix(start_pos, coin_pos, monsters_pos, fires_pos):
    coin_pos = check_coin(coin_pos, monsters_pos, fires_pos)

    tem_pos = coin_pos
    tem_pos.insert(0, start_pos)
    tem_dif_y = []
    for i in range(len(coin_pos)):
        tem_dif_x = []
        for j in range(len(coin_pos)):
            if (i == j):
                tem_dif_x.append(-1)
            else:
                tem_dif_x.append(abs(coin_pos[i][0] - coin_pos[j][0]) + abs(coin_pos[i][1] - coin_pos[j][1]))
        tem_dif_y.append(tem_dif_x)
    return tem_dif_y


def check_coin(coin_pos, monsters_pos, fires_pos):
    monsters_pos.append([-10, -10])
    fires_pos.append([-10, -10])

    tem_coin_pos = coin_pos
    p = 0.3  # parameter to adjust the closeness between coin to monster (or fire)
    for i in range(len(coin_pos)):
        for j in range(len(monsters_pos)):
            for k in range(len(fires_pos)):
                try:
                    if ((abs(coin_pos[i][0] - monsters_pos[j][0]) + abs(coin_pos[i][1] - monsters_pos[j][1])) < p) \
                            or ((abs(coin_pos[i][0] - fires_pos[k][0]) + abs(coin_pos[i][1] - fires_pos[k][1])) < p):
                        tem_coin_pos.pop(i)
                except:  # noqa: E722  (kept as in the original)
                    None
    return tem_coin_pos


def findMinRoute(tsp):
    sum = 0
    counter = 0
    j = 0
    i = 0
    min = INT_MAX
    visitedRouteList = DefaultDict(int)

    visitedRouteList[0] = 1
    route = [0] * len(tsp)

    while i < len(tsp) and j < len(tsp[i]):
        if counter >= len(tsp[i]) - 1:
            break
        if j != i and (visitedRouteList[j] == 0):
            if tsp[i][j] < min:
                min = tsp[i][j]
                route[counter] = j + 1
        j += 1
        if j == len(tsp[i]):
            sum += min
            min = INT_MAX
            visitedRouteList[route[counter] - 1] = 1
            j = 0
            i = route[counter] - 1
            counter += 1

    i = route[counter - 1] - 1
    for j in range(len(tsp)):
        if (i != j) and tsp[i][j] < min:
            min = tsp[i][j]
            route[counter] = j + 1
    sum += min
    return route
# ---- end of copied code ----


def _corners(obs):
    # position_func_v3 gives pos_hint (bottom-left corner), not the centre
    coin_pos = [[x - COIN_SIZE / 2.0, y - COIN_SIZE / 2.0] for x, y in obs.coins]
    monsters_pos = [[m[0] - MONSTER_SIZE / 2.0, m[1] - MONSTER_SIZE / 2.0] for m in obs.monsters]
    fires_pos = [[f.x - FIRE_SIZE / 2.0, f.y - FIRE_SIZE / 2.0] for f in obs.fires]
    return coin_pos, monsters_pos, fires_pos


class GreedyAgent:
    # same steps as CollectCoinThread.run()
    # act() returns (where to walk, how many frames to wait)

    name = "greedy"

    def __init__(self, seed=0):
        self._plan = None          # remaining stops of the current tour
        self._started = False

    def act(self, obs):
        if not self._started:
            # the original moves to start_pos first but doesn't sleep,
            # so the first coin move replaces it straight away
            self._started = True
        while True:
            if self._plan:
                target = self._plan.pop(0)
                return (float(target[0]), float(target[1])), int(round(SLEEP_SECONDS * FPS))
            if not obs.coins:
                return None, 1
            coin_pos, monsters_pos, fires_pos = _corners(obs)
            start_pos = [0.1, 0.1]
            tsp = create_matrix(start_pos, coin_pos, monsters_pos, fires_pos)
            route = findMinRoute(tsp)
            # create_matrix inserted start_pos into coin_pos, same as the original
            stops = [coin_pos[route[i] - 1] for i in range(0, len(route) - 1)]
            if not stops:
                # original just loops without moving until the hazards move away
                return None, 1
            self._plan = stops

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/agents/greedy.py


In [8]:
%%writefile "{PKG}/agents/ga_task2.py"
"""
Our GA agent from Lab Task 2 (Group 01), not changed.

Same fitness (evaluate_target), same move limit (feasible_target), same GA
settings as our submitted main_updated.py. Every 0.25 s pygad evolves a
target (x, y) for 8 generations and keeps the population for the next move.
The only difference is the input: the game made a WorldSnapshot from the
Kivy widgets, here we make it from the simulator.
"""

import collections
import math

import pygad

COIN_CLUSTER_K = 3
COIN_REWARD_SCALE = 1.0
COIN_DIST_OFFSET = 0.1
TRAVEL_PENALTY = 0.8

MONSTER_PENALTY = 15.0
MONSTER_DANGER_RADIUS = 0.40
FIRE_PENALTY = 15.0
FIRE_DANGER_RADIUS = 0.35

MOVE_SECONDS = 0.25
MAX_STEP = 0.25
PATH_SAMPLES = 8

STAGNATION_GAIN = 0.6
STAGNATION_CAP = 6.0
OUT_OF_BOUNDS_PENALTY = 5.0

POPULATION = 20
PLAN_GENERATIONS = 8
MUTATION_STEP = 0.2

WorldSnapshot = collections.namedtuple(
    "WorldSnapshot", "level token char coins monsters fires margin stagnation")


def _manhattan(ax, ay, bx, by):
    return abs(ax - bx) + abs(ay - by)


def feasible_target(tx, ty, snapshot):
    half_w, half_h = snapshot.margin
    tx = min(max(tx, half_w), 1.0 - half_w)
    ty = min(max(ty, half_h), 1.0 - half_h)
    char_x, char_y = snapshot.char
    dx, dy = tx - char_x, ty - char_y
    length = math.hypot(dx, dy)
    if length > MAX_STEP:
        scale = MAX_STEP / length
        tx, ty = char_x + dx * scale, char_y + dy * scale
    return tx, ty


def _path_hazard_penalty(snapshot, tx, ty, hazards, penalty, radius):
    if not hazards:
        return 0.0
    char_x, char_y = snapshot.char
    total = 0.0
    for step in range(1, PATH_SAMPLES + 1):
        fraction = step / float(PATH_SAMPLES)
        px = char_x + (tx - char_x) * fraction
        py = char_y + (ty - char_y) * fraction
        moment = MOVE_SECONDS * fraction
        for hx, hy, vx, vy in hazards:
            distance = _manhattan(px, py, hx + vx * moment, hy + vy * moment)
            if distance < radius:
                total += penalty * (radius - distance) / radius
    return total / PATH_SAMPLES


def evaluate_target(tx, ty, snapshot):
    coins = snapshot.coins
    if not coins:
        return 0.0
    fx, fy = feasible_target(tx, ty, snapshot)
    distances = sorted(_manhattan(fx, fy, cx, cy) for cx, cy in coins)
    reward = 0.0
    for distance in distances[:COIN_CLUSTER_K]:
        reward += COIN_REWARD_SCALE / (distance + COIN_DIST_OFFSET)
    reward *= min(1.0 + STAGNATION_GAIN * snapshot.stagnation, STAGNATION_CAP)
    char_x, char_y = snapshot.char
    score = reward - TRAVEL_PENALTY * _manhattan(fx, fy, char_x, char_y)
    score -= _path_hazard_penalty(snapshot, fx, fy, snapshot.monsters,
                                  MONSTER_PENALTY, MONSTER_DANGER_RADIUS)
    score -= _path_hazard_penalty(snapshot, fx, fy, snapshot.fires,
                                  FIRE_PENALTY, FIRE_DANGER_RADIUS)
    half_w, half_h = snapshot.margin
    for value, low, high in ((tx, half_w, 1.0 - half_w), (ty, half_h, 1.0 - half_h)):
        if value < low:
            score -= OUT_OF_BOUNDS_PENALTY * (low - value)
        elif value > high:
            score -= OUT_OF_BOUNDS_PENALTY * (value - high)
    return score


def to_snapshot(obs):
    return WorldSnapshot(
        level=obs.level, token=obs.frame, char=obs.char, coins=obs.coins,
        monsters=obs.monsters,
        fires=tuple((f.x, f.y, f.vx, f.vy) for f in obs.fires),
        margin=(obs.margin, obs.margin), stagnation=obs.stagnation)


class GAAgent:
    # same as CollectCoinThread.plan() in Task 2

    name = "ga_task2"

    def __init__(self, seed=0):
        self.seed = int(seed)
        self.steps = 0
        self.population = None

    def act(self, obs):
        from ..sim import MOVE_FRAMES
        snapshot = to_snapshot(obs)
        if not snapshot.coins:
            return None, 1

        def fitness_func(ga_instance, solution, solution_idx):
            return evaluate_target(float(solution[0]), float(solution[1]), snapshot)

        # seed each step so runs can be repeated
        seed = (self.seed * 7919 + self.steps) % (2 ** 31 - 1)
        self.steps += 1
        kwargs = dict(num_generations=PLAN_GENERATIONS,
                      num_parents_mating=4,
                      fitness_func=fitness_func,
                      parent_selection_type="tournament",
                      K_tournament=3,
                      keep_elitism=2,
                      sol_per_pop=POPULATION,
                      num_genes=2,
                      gene_space={'low': 0.0, 'high': 1.0},
                      init_range_low=0.0,
                      init_range_high=1.0,
                      crossover_type="single_point",
                      mutation_type="random",
                      mutation_percent_genes=50,
                      random_mutation_min_val=-MUTATION_STEP,
                      random_mutation_max_val=MUTATION_STEP,
                      mutation_by_replacement=False,
                      random_seed=seed,
                      suppress_warnings=True,
                      save_solutions=False,
                      save_best_solutions=False)
        if self.population is not None:
            kwargs["initial_population"] = self.population
        ga_instance = pygad.GA(**kwargs)
        ga_instance.run()
        self.population = ga_instance.population
        best, _, _ = ga_instance.best_solution(ga_instance.last_generation_fitness)
        return feasible_target(float(best[0]), float(best[1]), snapshot), MOVE_FRAMES

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/agents/ga_task2.py


In [9]:
%%writefile "{PKG}/agents/proposed.py"
"""
Our improved agent: RH-STA* (risk-aware receding-horizon space-time A*).

The idea:
1. Coin order - nearest neighbour tour + 2-opt, starting from where the
   character actually is (the original always started from (0.1, 0.1)).
2. Path - A* over (x, y, t). t = number of 0.25 s moves ahead. From each
   state we can move in 16 directions with 3 step sizes, step onto the goal
   coin, or wait. Every move is checked against where the hazards WILL be:
     - fires move on fixed paths (from cointex.kv) so we can predict them exactly
     - monsters are predicted with their current velocity, plus a safety
       margin that grows the further ahead we look (they can turn at any time)
   If a move hits a hazard in the first few steps we don't allow it. Later
   hits and near misses just add cost.
3. We only do the first move of the plan, then look again and re-plan
   (every 0.25 s).

Because time is part of the state, A* can choose to wait for a fire to pass
and then grab the coin. Greedy and the GA can't do that.

cost = 1 per move + 0.2 * distance + risk_weight * near-miss risk
h    = distance left / max step  (number of moves still needed)
"""

import heapq
import math
import time
from dataclasses import dataclass, replace

import numpy as np

from ..sim import COIN_GAB, FIRE_GAB, MONSTER_GAB, MONSTER_SIZE, MOVE_FRAMES, MOVE_SECONDS

SQRT2 = math.sqrt(2.0)


@dataclass(frozen=True)
class Config:
    name: str = "proposed"
    # moves
    max_step: float = 0.30          # longest move in one 0.25 s step
    step_fracs: tuple = (1.0, 0.5, 0.25)
    n_dirs: int = 16
    # search
    horizon: int = 8                # how many moves ahead (8 x 0.25 = 2 s)
    max_expansions: int = 100      # total for one move, shared by all the goals we try
    heuristic_weight: float = 1.2   # a bit greedy so A* is faster
    samples: int = 8                # points checked along each move
    grid: float = 0.02              # for the closed set
    # hazards
    use_hazards: bool = True
    predict_motion: bool = True
    monster_margin: float = 0.05
    fire_margin: float = 0.035
    monster_sigma: float = 0.10     # extra margin per second we look ahead
    contact_buffer: float = 0.01
    hard_steps: int = 3             # no predicted contact allowed in the first 3 moves
    late_contact_penalty: float = 3.0
    risk_weight: float = 4.0
    dist_weight: float = 0.2
    # goals
    use_tour: bool = True
    two_opt: bool = True
    alt_goals: int = 3
    defer_seconds: float = 1.0
    # if we're stuck (no coin for a while) we accept more risk
    stuck_seconds: float = 4.0
    stagnation_relax: float = 0.25


DEFAULT = Config()

# versions with one part turned off, for the ablation test
VARIANTS = {
    "proposed": DEFAULT,
    "proposed_no_hazard": replace(DEFAULT, name="proposed_no_hazard", use_hazards=False),
    "proposed_static_hazard": replace(DEFAULT, name="proposed_static_hazard", predict_motion=False),
    "proposed_no_tour": replace(DEFAULT, name="proposed_no_tour", use_tour=False),
}


def nn_two_opt_tour(start, coins, two_opt=True):
    # nearest neighbour tour from start, then 2-opt to remove crossings.
    # open path (we don't come back to the start). returns coin indexes in order
    n = len(coins)
    if n <= 1:
        return list(range(n))
    pts = np.vstack([np.asarray(start)[None, :], np.asarray(coins)])
    dist = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1))
    order = [0]
    unvisited = set(range(1, n + 1))
    while unvisited:
        last = order[-1]
        nxt = min(unvisited, key=lambda j: dist[last, j])
        order.append(nxt)
        unvisited.remove(nxt)
    if two_opt:
        improved = True
        while improved:
            improved = False
            for i in range(1, n):
                for j in range(i + 1, n + 1):
                    a, b = order[i - 1], order[i]
                    c = order[j]
                    d = order[j + 1] if j + 1 <= n else None
                    before = dist[a, b] + (dist[c, d] if d is not None else 0.0)
                    after = dist[a, c] + (dist[b, d] if d is not None else 0.0)
                    if after < before - 1e-9:
                        order[i:j + 1] = order[i:j + 1][::-1]
                        improved = True
    return [k - 1 for k in order[1:]]


def tour_length(start, coins, order):
    pts = [np.asarray(start)] + [np.asarray(coins[k]) for k in order]
    return float(sum(np.hypot(*(pts[i + 1] - pts[i])) for i in range(len(pts) - 1)))


def _fire_positions(f, taus):
    # where a fire will be after each time in taus: finish the current sweep,
    # then go back and forth between the two end points
    a = np.array([f.x, f.y])
    b = np.array([f.tx, f.ty])
    c = np.array([f.ox, f.oy])
    r, D = max(f.remaining, 1e-9), f.leg_duration
    t = taus[..., None]
    first = a + (b - a) * np.clip(t / r, 0.0, 1.0)
    tp = np.maximum(t - r, 0.0)
    n = np.floor(tp / D)
    ph = (tp - n * D) / D
    forward = (n % 2 == 0)
    later = np.where(forward, b + (c - b) * ph, c + (b - c) * ph)
    return np.where(t <= r, first, later)


class ProposedAgent:
    def __init__(self, seed=0, config=DEFAULT):
        self.cfg = config
        self.name = config.name
        self._tour = []
        self._tour_key = None
        self._deferred = {}         # coin -> time we skip it until
        self.plan_ms = []
        self.last_plan = None       # kept so we can draw the plan
        c = config
        angles = np.arange(c.n_dirs) * (2 * math.pi / c.n_dirs)
        unit = np.stack([np.cos(angles), np.sin(angles)], axis=1)
        # all the moves we try, last row is "wait"
        self._moves = np.concatenate([unit * (c.max_step * s) for s in c.step_fracs]
                                     + [np.zeros((1, 2))])
        self._fracs = np.arange(1, c.samples + 1) / c.samples

    def _predict(self, obs):
        # predict all hazard positions for every move and sample point up to the horizon
        c = self.cfg
        taus = (np.arange(c.horizon)[:, None] + self._fracs[None, :]) * MOVE_SECONDS   # (H, S)
        pos, gab, margin = [], [], []
        if c.use_hazards:
            lo, hi = MONSTER_SIZE / 2.0, 1.0 - MONSTER_SIZE / 4.0 + MONSTER_SIZE / 2.0
            for x, y, vx, vy in obs.monsters:
                if c.predict_motion:
                    p = np.stack([np.clip(x + vx * taus, lo, hi), np.clip(y + vy * taus, lo, hi)], -1)
                else:
                    p = np.broadcast_to(np.array([x, y]), taus.shape + (2,))
                pos.append(p)
                gab.append(MONSTER_GAB)
                margin.append(c.monster_margin + c.monster_sigma * taus)
            for f in obs.fires:
                if c.predict_motion:
                    p = _fire_positions(f, taus)
                else:
                    p = np.broadcast_to(np.array([f.x, f.y]), taus.shape + (2,))
                pos.append(p)
                gab.append(FIRE_GAB)
                margin.append(np.full(taus.shape, c.fire_margin))
        if pos:
            self._hz = np.stack(pos, axis=2)     # (H, S, M, 2)
            self._gab = np.array(gab)            # (M,)
            self._mg = np.stack(margin, axis=2)  # (H, S, M)
        else:
            self._hz = None

    def _edges(self, p, succ, k):
        # risk + contact for every move p -> succ during move number k
        # (numpy so we check all moves at once)
        pts = p[None, None, :] + (succ - p)[:, None, :] * self._fracs[None, :, None]   # (A, S, 2)
        if self._hz is None:
            zeros = np.zeros(len(succ))
            return zeros, zeros.astype(bool), pts
        # same box-style distance the game uses for collisions
        d = np.abs(pts[:, :, None, :] - self._hz[k][None]).max(-1)
        clearance = d - self._gab[None, None, :]
        contact = (clearance <= self.cfg.contact_buffer).any(axis=(1, 2))
        risk = np.clip(1.0 - clearance / self._mg[k][None], 0.0, 1.0).sum(axis=(1, 2)) / self.cfg.samples
        return risk, contact, pts

    def _successors(self, p, goal, margin):
        c = self.cfg
        to_goal = goal - p
        dist = math.hypot(*to_goal)
        direct = goal if dist <= c.max_step else p + to_goal * (c.max_step / dist)
        succ = np.vstack([p + self._moves, direct[None, :]])
        return np.clip(succ, margin, 1.0 - margin)

    def _astar(self, obs, start, goal, risk_weight, hard_steps, budget):
        c = self.cfg
        H = c.horizon
        reach = COIN_GAB * 0.9
        h0 = max(0.0, math.hypot(*(goal - start)) - COIN_GAB * SQRT2) / c.max_step
        # node info stored in lists, index = node id
        xs, ys, ks, gs, parents, hs = [start[0]], [start[1]], [0], [0.0], [-1], [h0]
        heap = [(c.heuristic_weight * h0, 0.0, 0, False)]
        closed = set()
        best, best_key = None, None
        expansions = 0
        tie = 0
        while heap and expansions < budget:
            _, _, i, reached = heapq.heappop(heap)
            if reached:
                return self._path(i, parents, xs, ys), gs[i], True, expansions
            k = ks[i]
            key = (int(round(xs[i] / c.grid)), int(round(ys[i] / c.grid)), k)
            if key in closed:
                continue
            closed.add(key)
            expansions += 1
            # remember the node closest to the goal in case we don't reach it
            if i != 0:
                score = (hs[i], gs[i])
                if best_key is None or score < best_key:
                    best, best_key = i, score
            if k >= H:
                continue
            p = np.array([xs[i], ys[i]])
            succ = self._successors(p, goal, obs.margin)
            risk, contact, pts = self._edges(p, succ, k)
            lengths = np.hypot(*(succ - p).T)
            cost = 1.0 + c.dist_weight * lengths + risk_weight * risk
            if k >= hard_steps:
                cost = cost + c.late_contact_penalty * contact
                allowed = np.ones(len(succ), dtype=bool)
            else:
                allowed = ~contact
            hit_goal = (np.abs(pts - goal[None, None, :]).max(-1) <= reach).any(axis=1)
            dist_goal = np.hypot(*(succ - goal).T)
            hvals = np.where(hit_goal, 0.0, np.maximum(0.0, dist_goal - COIN_GAB * SQRT2) / c.max_step)
            g0 = gs[i]
            for a in np.nonzero(allowed)[0]:
                j = len(xs)
                xs.append(float(succ[a, 0]))
                ys.append(float(succ[a, 1]))
                ks.append(k + 1)
                g = g0 + float(cost[a])
                gs.append(g)
                parents.append(i)
                hs.append(float(hvals[a]))
                tie += 1
                heapq.heappush(heap, (g + c.heuristic_weight * hvals[a], tie, j, bool(hit_goal[a])))
        if best is None:
            return None, math.inf, False, expansions
        return self._path(best, parents, xs, ys), gs[best], False, expansions

    @staticmethod
    def _path(i, parents, xs, ys):
        path = []
        while i > 0:
            path.append((xs[i], ys[i]))
            i = parents[i]
        return path[::-1]

    def _escape(self, obs, start, goal):
        # every move is dangerous: pick the one that keeps us furthest from hazards
        succ = self._successors(start, goal, obs.margin)
        if self._hz is None:
            return tuple(succ[-1])
        pts = start[None, None, :] + (succ - start)[:, None, :] * self._fracs[None, :, None]
        d = np.abs(pts[:, :, None, :] - self._hz[0][None]).max(-1) - self._gab[None, None, :]
        worst = d.min(axis=(1, 2))
        progress = -np.hypot(*(succ - goal).T) * 1e-3
        return tuple(succ[int(np.argmax(worst + progress))])

    def _goal_order(self, obs):
        coins = obs.coins
        now = obs.time
        self._deferred = {k: t for k, t in self._deferred.items() if t > now and k in coins}
        if not self.cfg.use_tour:
            cx, cy = obs.char
            order = sorted(range(len(coins)), key=lambda j: math.hypot(coins[j][0] - cx, coins[j][1] - cy))
        else:
            # only rebuild the tour when the coins change
            key = (coins, tuple(sorted(self._deferred)))
            if key != self._tour_key:
                order = nn_two_opt_tour(obs.char, coins, self.cfg.two_opt)
                self._tour = [coins[j] for j in order]
                self._tour_key = key
            order = [coins.index(cn) for cn in self._tour]
        # skipped coins go to the back
        free = [j for j in order if coins[j] not in self._deferred]
        late = [j for j in order if coins[j] in self._deferred]
        return free + late

    def act(self, obs):
        if not obs.coins:
            return None, 1
        t0 = time.perf_counter()
        c = self.cfg
        self._predict(obs)
        start = np.array(obs.char)
        stuck = max(0.0, obs.stagnation - c.stuck_seconds)
        risk_weight = c.risk_weight / (1.0 + c.stagnation_relax * stuck)
        hard_steps = c.hard_steps if (stuck < c.stuck_seconds or obs.life < 0.5) else 1

        order = self._goal_order(obs)
        best_partial = None
        target = None
        # try the next coin in the tour, if blocked try the next 2
        # all tries share one search budget so a move never takes too long
        budget = c.max_expansions
        for rank, j in enumerate(order[:max(1, c.alt_goals)]):
            if rank > 0 and budget < 20:
                break
            goal = np.array(obs.coins[j])
            path, cost, reached, used = self._astar(obs, start, goal, risk_weight, hard_steps, budget)
            budget -= used
            if reached and path:
                target, self.last_plan = path[0], path
                break
            if rank == 0 and path:
                best_partial = path
            if c.use_tour and rank == 0:
                self._deferred[obs.coins[j]] = obs.time + c.defer_seconds
        if target is None:
            if best_partial:
                target, self.last_plan = best_partial[0], best_partial
            else:
                target = self._escape(obs, start, np.array(obs.coins[order[0]]))
                self.last_plan = [target]
        self.plan_ms.append((time.perf_counter() - t0) * 1000.0)
        return (float(target[0]), float(target[1])), MOVE_FRAMES

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/agents/proposed.py


In [10]:
%%writefile "{PKG}/runner.py"
"""
Runs the agents on the levels and saves the results.

All agents get the same (level, seed) pairs so we can compare them fairly.
Results go into a csv one row at a time, and rows that are already there
get skipped, so if Colab disconnects we just run the cell again.
"""

import csv
import hashlib
import os
import shutil
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

from . import sim
from .agents.ga_task2 import GAAgent
from .agents.greedy import GreedyAgent
from .agents import proposed as _proposed
from .agents.proposed import VARIANTS, ProposedAgent

FIELDS = ["level", "seed", "agent", "hazard_speed", "won", "died", "timeout",
          "coins_total", "coins_collected", "coin_pct",
          "seconds", "completion_time", "remaining_life",
          "monster_hit_frames", "fire_hit_frames", "near_miss_pct",
          "path_length", "racr", "decisions", "mean_plan_ms", "max_plan_ms",
          "wall_seconds"]


def make_agent(name, seed, config=None):
    if name == "greedy":
        return GreedyAgent(seed)
    if name == "ga_task2":
        return GAAgent(seed)
    if config is not None:
        return ProposedAgent(seed, config)
    if name in VARIANTS:
        return ProposedAgent(seed, VARIANTS[name])
    raise KeyError(name)


def play_episode(level, seed, agent_name, config=None, record=False,
                 time_limit=sim.TIME_LIMIT_SECONDS, hazard_speed=1.0):
    # play one level until win / death / timeout and return the results
    # (record=True also keeps every frame so we can draw it or make a gif)
    env = sim.CoinTexSim(level, seed, time_limit=time_limit, hazard_speed=hazard_speed)
    agent = make_agent(agent_name, seed, config)
    trace = {"char": [env.agent.copy()], "monsters": [], "fires": [], "coins_left": [],
             "coins": [], "plans": []} if record else None
    plan_ms = []
    decisions = 0
    wall0 = time.perf_counter()
    while not env.done:
        obs = env.observe()
        t0 = time.perf_counter()
        target, wait_frames = agent.act(obs)
        plan_ms.append((time.perf_counter() - t0) * 1000.0)
        decisions += 1
        env.command(target)
        for _ in range(max(1, int(wait_frames))):
            if env.done:
                break
            env.step_frame()
            if record:
                trace["char"].append(env.agent.copy())
                trace["monsters"].append([m["pos"].copy() for m in env.monsters])
                trace["fires"].append([f["pos"].copy() for f in env.fires])
                trace["coins_left"].append(len(env.coins))
                trace["coins"].append([c.copy() for c in env.coins])
        if record and getattr(agent, "last_plan", None) is not None:
            trace["plans"].append((env.frames, list(agent.last_plan)))

    coins_collected = env.num_coins - len(env.coins)
    life = env.remaining_life_fraction()
    secs = env.seconds
    row = {
        "level": level, "seed": seed, "agent": agent_name if config is None else config.name,
        "hazard_speed": hazard_speed,
        "won": int(env.won), "died": int(env.killed), "timeout": int(env.timed_out),
        "coins_total": env.num_coins, "coins_collected": coins_collected,
        "coin_pct": round(100.0 * coins_collected / env.num_coins, 2),
        "seconds": round(secs, 3),
        "completion_time": round(secs, 3) if env.won else float("nan"),
        "remaining_life": round(100.0 * life, 2),
        "monster_hit_frames": env.monster_hit_frames,
        "fire_hit_frames": env.fire_hit_frames,
        "near_miss_pct": round(100.0 * env.near_miss_frames / max(env.frames, 1), 2),
        "path_length": round(env.path_length, 3),
        # our extra metric (RACR): coins per second x life left
        "racr": round((coins_collected / max(secs, 1e-9)) * life, 4),
        "decisions": decisions,
        "mean_plan_ms": round(float(np.mean(plan_ms)), 3) if plan_ms else 0.0,
        "max_plan_ms": round(float(np.max(plan_ms)), 3) if plan_ms else 0.0,
        "wall_seconds": round(time.perf_counter() - wall0, 3),
    }
    if record:
        trace["env"] = env
        trace["coin_times"] = list(env.coin_times)
        return row, trace
    return row


def _job(args):
    level, seed, agent_name, config, speed = args
    return play_episode(level, seed, agent_name, config=config, hazard_speed=speed)


def _done_keys(path):
    # which runs are already in the csv
    if not os.path.exists(path):
        return set()
    with open(path, newline="") as f:
        return {(int(r["level"]), int(r["seed"]), r["agent"], float(r["hazard_speed"]))
                for r in csv.DictReader(f)}


def run_sweep(csv_path, levels, seeds, agents, configs=None, workers=None, verbose=True,
              hazard_speeds=(1.0,)):
    # agents  = list of agent names ("greedy", "ga_task2", "proposed", ...)
    # configs = extra proposed settings to try, {name: Config} (for the sensitivity test)
    configs = configs or {}
    os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
    done = _done_keys(csv_path)
    jobs = []
    for speed in hazard_speeds:
        speed = float(speed)
        for level in levels:
            for seed in seeds:
                for name in agents:
                    if (level, seed, name, speed) not in done:
                        jobs.append((level, seed, name, None, speed))
                for name, cfg in configs.items():
                    if (level, seed, name, speed) not in done:
                        jobs.append((level, seed, name, cfg, speed))
    total = len(jobs)
    if verbose:
        print(f"{len(done)} episodes already in {os.path.basename(csv_path)}; running {total} more")
    if not total:
        return
    new_file = not os.path.exists(csv_path)
    workers = workers or max(1, (os.cpu_count() or 1))
    t0 = time.time()
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            writer.writeheader()
        finished = 0

        def report(row):
            nonlocal finished
            writer.writerow(row)
            f.flush()
            finished += 1
            if verbose and (finished % max(1, total // 20) == 0 or finished == total):
                rate = finished / max(time.time() - t0, 1e-9)
                eta = (total - finished) / max(rate, 1e-9)
                print(f"  {finished}/{total} episodes  ({rate:.2f}/s, ETA {eta/60:.1f} min)")

        # run in parallel on all cpu cores
        if workers == 1:
            for job in jobs:
                report(_job(job))
        else:
            with ProcessPoolExecutor(max_workers=workers) as pool:
                futures = [pool.submit(_job, job) for job in jobs]
                for fut in as_completed(futures):
                    report(fut.result())


def summarise(df, by=("agent",)):
    # averages per agent (or per agent + level) for the tables
    import pandas as pd
    g = df.groupby(list(by))
    out = pd.DataFrame({
        "episodes": g.size(),
        "success_rate": 100.0 * g["won"].mean(),
        "death_rate": 100.0 * g["died"].mean(),
        "timeout_rate": 100.0 * g["timeout"].mean(),
        "avg_coin_pct": g["coin_pct"].mean(),
        "completion_time_mean": g["completion_time"].mean(),
        "completion_time_median": g["completion_time"].median(),
        "episode_time_mean": g["seconds"].mean(),
        "remaining_life_mean": g["remaining_life"].mean(),
        "near_miss_pct_mean": g["near_miss_pct"].mean(),
        "racr_mean": g["racr"].mean(),
        "path_length_mean": g["path_length"].mean(),
        "plan_ms_mean": g["mean_plan_ms"].mean(),
    })
    return out.reset_index()


def proposed_code_tag():
    # changes whenever proposed.py changes
    with open(_proposed.__file__, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()[:8]


def reset_if_changed(csv_path, tag):
    # if the csv was made with an older version of our agent, move it to *_old_<tag>.csv
    # so the sweep runs again instead of reusing old rows
    tag_file = csv_path + ".tag"
    old = open(tag_file).read().strip() if os.path.exists(tag_file) else None
    if os.path.exists(csv_path) and old != tag:
        backup = csv_path.replace(".csv", f"_old_{old or 'v1'}.csv")
        shutil.move(csv_path, backup)
        print("agent code changed - old results moved to", os.path.basename(backup))
    os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
    with open(tag_file, "w") as f:
        f.write(tag)

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/runner.py


In [11]:
%%writefile "{PKG}/plots.py"
"""
Plotting functions for the notebooks and report.
Each agent always has the same colour.
"""

import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from . import sim

AGENT_COLORS = {
    "proposed": "#2a78d6",                 # blue
    "greedy": "#eb6834",                   # orange
    "ga_task2": "#1baf7a",                 # aqua
    "proposed_no_hazard": "#eda100",       # yellow
    "proposed_static_hazard": "#e87ba4",   # magenta
    "proposed_no_tour": "#6250d6",         # violet
}
AGENT_LABELS = {
    "proposed": "Proposed (RH-STA*)",
    "greedy": "Original greedy",
    "ga_task2": "GA (Task 2)",
    "proposed_no_hazard": "RH-STA* w/o hazards",
    "proposed_static_hazard": "RH-STA* static hazards",
    "proposed_no_tour": "RH-STA* w/o 2-opt tour",
}
INK, MUTED, GRID = "#1f2328", "#59636e", "#d0d7de"


def style():
    matplotlib.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
        "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titleweight": "bold",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
        "legend.frameon": False, "font.size": 10,
    })


def color(agent):
    return AGENT_COLORS.get(agent, "#59636e")


def label(agent):
    return AGENT_LABELS.get(agent, agent)


def save(fig, folder, name):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    fig.savefig(path)
    return path


def per_level_bars(df, metric, agents, ylabel, title, agg="mean", ylim=None, fmt="%.0f",
                   scale=1.0, figsize=(14, 4.6)):
    # bars per level, one bar per agent
    levels = sorted(df["level"].unique())
    x = np.arange(len(levels))
    width = 0.84 / len(agents)
    fig, ax = plt.subplots(figsize=figsize)
    for i, agent in enumerate(agents):
        sub = df[df["agent"] == agent].groupby("level")[metric].agg(agg).reindex(levels) * scale
        ax.bar(x + (i - (len(agents) - 1) / 2) * width, sub.values, width * 0.92,
               color=color(agent), label=label(agent), edgecolor="white", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([f"L{lv}" for lv in levels])
    ax.set_xlabel("Level")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis="x", visible=False)
    ax.legend(ncol=len(agents), loc="upper center", bbox_to_anchor=(0.5, -0.14))
    fig.tight_layout()
    return fig


def overall_bars(summary, metrics, agents, titles, fmts=None, figsize=None):
    # one small chart per metric with the value on each bar
    fmts = fmts or ["%.1f"] * len(metrics)
    figsize = figsize or (3.4 * len(metrics), 3.6)
    fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
    axes = np.atleast_1d(axes)
    s = summary.set_index("agent").reindex(agents)
    for ax, metric, title, fmt in zip(axes, metrics, titles, fmts):
        vals = s[metric].values
        bars = ax.bar(range(len(agents)), vals, 0.7, color=[color(a) for a in agents],
                      edgecolor="white", linewidth=0.6)
        ax.bar_label(bars, fmt=fmt, padding=2, fontsize=9, color=INK)
        ax.set_xticks(range(len(agents)))
        ax.set_xticklabels([label(a).replace(" (", "\n(") for a in agents], fontsize=8)
        ax.set_title(title, loc="left", fontsize=10)
        ax.grid(axis="x", visible=False)
        top = np.nanmax(vals) if np.isfinite(vals).any() else 1
        ax.set_ylim(0, top * 1.18 if top > 0 else 1)
    fig.tight_layout()
    return fig


def success_heatmap(df, agents, title="Success rate (%) by level"):
    levels = sorted(df["level"].unique())
    mat = np.array([[100 * df[(df.agent == a) & (df.level == lv)]["won"].mean() for lv in levels]
                    for a in agents])
    fig, ax = plt.subplots(figsize=(14, 0.55 * len(agents) + 1.4))
    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    for i in range(len(agents)):
        for j in range(len(levels)):
            v = mat[i, j]
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 60 else INK)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([f"L{lv}" for lv in levels])
    ax.set_yticks(range(len(agents)))
    ax.set_yticklabels([label(a) for a in agents])
    ax.grid(False)
    ax.set_title(title, loc="left")
    fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
    fig.tight_layout()
    return fig


def completion_box(df, agents, title="Completion time of won episodes"):
    fig, ax = plt.subplots(figsize=(8, 4))
    data = [df[(df.agent == a) & (df.won == 1)]["completion_time"].dropna().values for a in agents]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5, showfliers=False,
                    medianprops=dict(color=INK, linewidth=1.5))
    for patch, a in zip(bp["boxes"], agents):
        patch.set_facecolor(color(a))
        patch.set_alpha(0.85)
        patch.set_edgecolor("white")
    ax.set_xticks(range(1, len(agents) + 1))
    ax.set_xticklabels([label(a) for a in agents])
    ax.set_ylabel("Seconds")
    ax.set_title(title, loc="left")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    return fig


def difficulty_curve(df, agents, metric="won", ylabel="Success rate (%)", scale=100.0,
                     title="Success rate vs level"):
    levels = sorted(df["level"].unique())
    fig, ax = plt.subplots(figsize=(11, 4))
    for a in agents:
        s = df[df.agent == a].groupby("level")[metric].mean().reindex(levels) * scale
        ax.plot(levels, s.values, "-o", color=color(a), lw=2, ms=5, label=label(a))
    ax.set_xticks(levels)
    ax.set_xlabel("Level")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.legend(ncol=len(agents), loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()
    return fig


def _outcome(env):
    return "WON" if env.won else ("DIED" if env.killed else "TIMEOUT")


def draw_trajectory(ax, trace, agent, frame=None):
    # character path + where the hazards ended + coins not collected
    env = trace["env"]
    path = np.array(trace["char"] if frame is None else trace["char"][:frame + 1])
    ax.plot(path[:, 0], path[:, 1], "-", color=color(agent), lw=1.6, zorder=2)
    ax.scatter(*path[0], c=INK, s=30, zorder=4)
    for coin in env.coins:
        ax.scatter(*coin, marker="o", c="#eda100", edgecolors=INK, s=60, zorder=3)
    for m in env.monsters:
        ax.add_patch(plt.Rectangle(m["pos"] - sim.MONSTER_GAB, 2 * sim.MONSTER_GAB, 2 * sim.MONSTER_GAB,
                                   color="#6250d6", alpha=0.25, zorder=1))
    for f in env.fires:
        ax.scatter(*f["pos"], marker="^", c="#e34948", s=70, zorder=3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_title(f"{label(agent)} - {_outcome(env)}\n"
                 f"{env.num_coins - len(env.coins)}/{env.num_coins} coins, "
                 f"life {env.remaining_life_fraction() * 100:.0f}%, {env.seconds:.1f}s", fontsize=10)


def trajectory_grid(traces, agents, suptitle):
    fig, axes = plt.subplots(1, len(agents), figsize=(5 * len(agents), 5.4))
    for ax, a in zip(np.atleast_1d(axes), agents):
        draw_trajectory(ax, traces[a], a)
    handles = [
        plt.Line2D([], [], marker="o", ls="", mfc="#eda100", mec=INK, label="coin left"),
        plt.Line2D([], [], marker="s", ls="", c="#6250d6", alpha=0.4, ms=10, label="monster (final)"),
        plt.Line2D([], [], marker="^", ls="", c="#e34948", label="fire (final)"),
        plt.Line2D([], [], marker="o", ls="", c=INK, label="start"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=4)
    fig.suptitle(suptitle, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    return fig


def animate_episode(trace, agent, path, fps=20, every=3, tail=90):
    # save a gif of one episode (draws every 3rd frame)
    from matplotlib.animation import FuncAnimation, PillowWriter
    env = trace["env"]
    chars = np.array(trace["char"])
    n = len(trace["monsters"])
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    line, = ax.plot([], [], "-", color=color(agent), lw=1.4)
    char = ax.add_patch(plt.Rectangle((0, 0), sim.COIN_GAB * 2, sim.COIN_GAB * 2,
                                      color=color(agent), alpha=0.9))
    mons = [ax.add_patch(plt.Rectangle((0, 0), 2 * sim.MONSTER_GAB, 2 * sim.MONSTER_GAB,
                                       color="#6250d6", alpha=0.3)) for _ in env.monsters]
    coins = ax.scatter([], [], marker="o", c="#eda100", edgecolors=INK, s=40)
    fires = ax.scatter([], [], marker="^", c="#e34948", s=60)
    title = ax.set_title("")
    frames = list(range(0, n, every))

    def update(i):
        c = chars[i + 1]
        lo = max(0, i + 1 - tail)
        line.set_data(chars[lo:i + 2, 0], chars[lo:i + 2, 1])
        char.set_xy(c - sim.COIN_GAB)
        for patch, m in zip(mons, trace["monsters"][i]):
            patch.set_xy(m - sim.MONSTER_GAB)
        coins.set_offsets(np.array(trace["coins"][i]).reshape(-1, 2))
        if trace["fires"][i]:
            fires.set_offsets(np.array(trace["fires"][i]))
        title.set_text(f"{label(agent)}  L{env.level_num}  t={i / sim.FPS:.1f}s  "
                       f"coins left {trace['coins_left'][i]}")
        return [line, char, coins, fires, title] + mons

    anim = FuncAnimation(fig, update, frames=frames, blit=False)
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return path

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/plots.py


In [12]:
%%writefile "{PKG}/selftest.py"
"""
Quick checks we run before the experiments (notebook 00).
Run with: python -m cointex_parta.selftest
"""

import math
import random

import numpy as np

from . import sim
from .agents import greedy
from .agents.proposed import _fire_positions, nn_two_opt_tour, tour_length
from .levels import LEVEL_TABLE
from .runner import play_episode


def check_level_table():
    assert len(LEVEL_TABLE) == 24
    for n, c in LEVEL_TABLE.items():
        assert c["num_coins"] > 0 and c["num_collisions_level"] > 0, n


def check_deterministic():
    a = play_episode(18, 3, "proposed")
    b = play_episode(18, 3, "proposed")
    for k in ("won", "seconds", "remaining_life", "coins_collected"):
        assert a[k] == b[k], (k, a[k], b[k])


def check_paired_worlds():
    # same seed should give the same coins and monster moves
    e1, e2 = sim.CoinTexSim(10, 7), sim.CoinTexSim(10, 7)
    assert all(np.allclose(p, q) for p, q in zip(e1.coins, e2.coins))
    e1.command((0.9, 0.9))
    for _ in range(300):
        e1.step_frame()
        e2.step_frame()
    for m1, m2 in zip(e1.monsters, e2.monsters):
        assert np.allclose(m1["pos"], m2["pos"])


def check_fire_prediction():
    # our fire prediction should match the simulator
    worst = 0.0
    for level in (14, 18, 19, 20):
        env = sim.CoinTexSim(level, 0)
        env.run_frames(random.Random(level).randint(30, 400))
        obs = env.observe()
        future = []
        for _ in range(120):                       # 2 s ahead
            env.step_frame()
            future.append([f["pos"].copy() for f in env.fires])
        taus = np.arange(1, 121) * sim.DT
        for j, f in enumerate(obs.fires):
            pred = _fire_positions(f, taus)
            true = np.array([fr[j] for fr in future])
            worst = max(worst, float(np.abs(pred - true).max()))
    assert worst < 1e-6, worst
    return f"max error {worst:.2e}"


def check_tour():
    rng = np.random.default_rng(0)
    for _ in range(20):
        coins = [tuple(p) for p in rng.random((15, 2))]
        start = (0.1, 0.1)
        nn = nn_two_opt_tour(start, coins, two_opt=False)
        opt = nn_two_opt_tour(start, coins, two_opt=True)
        assert sorted(opt) == list(range(15))
        assert tour_length(start, coins, opt) <= tour_length(start, coins, nn) + 1e-9


def check_greedy_verbatim():
    # 2 coins, no hazards: route should visit both coins
    tsp = greedy.create_matrix([0.1, 0.1], [[0.5, 0.5], [0.2, 0.2]], [], [])
    route = greedy.findMinRoute(tsp)
    assert sorted(route[:-1]) == [2, 3], route
    # check_coin bug: coins near the monster still survive because of the pop
    coins = [[0.1 * i, 0.5] for i in range(6)]
    kept = greedy.check_coin([c[:] for c in coins], [[0.25, 0.5]], [])
    assert len(kept) > 0


def run_all():
    ok = True
    for fn in (check_level_table, check_deterministic, check_paired_worlds,
               check_fire_prediction, check_tour, check_greedy_verbatim):
        try:
            note = fn()
            print(f"PASS  {fn.__name__}" + (f"  ({note})" if note else ""))
        except Exception as exc:
            ok = False
            print(f"FAIL  {fn.__name__}: {exc!r}")
    return ok


if __name__ == "__main__":
    raise SystemExit(0 if run_all() else 1)

Writing /content/drive/MyDrive/COMP6009_Assignment-Part A/code/cointex_parta/selftest.py


## 2. Quick tests
Before running the experiments we check that:
- the same seed gives the same world (so every agent gets the same coins and monster moves)
- our fire prediction matches the simulator
- 2-opt never makes the coin route longer
- the copied greedy functions still behave like the original

In [13]:
for name in list(sys.modules):
    if name.startswith('cointex_parta'):
        del sys.modules[name]
importlib.invalidate_caches()
import cointex_parta
from cointex_parta import selftest
assert selftest.run_all(), 'self-test failed'

PASS  check_level_table
PASS  check_deterministic
PASS  check_paired_worlds
PASS  check_fire_prediction  (max error 5.33e-15)
PASS  check_tour
PASS  check_greedy_verbatim


## 3. Check the levels against the game files
We took the monster speeds and fire paths for all 24 levels from `cointex.kv`, so here we check
them against a fresh clone of the repo.

Something we noticed: the coin counts in `main_work_greedy.py` (from Blackboard) are not the same as
the repo's `main.py` (levels 1, 17, 19, 20), and levels 20 and 22 have extra hazards in the kv file that
never move. We followed `main_work_greedy.py` since that is the assignment version.
If `main_work_greedy.py` is in `data/` we check the coin counts as well.

In [14]:
import shutil
if not os.path.exists('/content/CoinTex'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ahmedfgad/CoinTex.git', '/content/CoinTex'], check=True)
kv_path = '/content/CoinTex/PlayerGA/TutorialProject/cointex.kv'
shutil.copy(kv_path, f"{PATHS['data']}/cointex.kv")
kv_text = open(kv_path).read()

from cointex_parta import levels
problems = levels.verify_kv(kv_text)
print('cointex.kv vs simulator:', 'EXACT MATCH for all 24 levels' if not problems else problems)

greedy_file = f"{PATHS['data']}/main_work_greedy.py"
if os.path.exists(greedy_file):
    p2 = levels.diff_against(levels.parse_level_table(kv_text, open(greedy_file).read()))
    print('main_work_greedy.py vs simulator:', 'EXACT MATCH' if not p2 else p2)
else:
    print('main_work_greedy.py not in data/ - coin counts not checked')
assert not problems

cointex.kv vs simulator: EXACT MATCH for all 24 levels
main_work_greedy.py not in data/ - coin counts not checked


In [15]:
import pandas as pd
table = pd.DataFrame([levels.level_summary(n) for n in range(1, 25)])
table.to_csv(f"{PATHS['data']}/level_table.csv", index=False)
table.T

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
level,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
coins,5,8,12,10,15,12,10,15,12,14,...,16,15,10,15,12,15,18,20,25,20
monsters,1,1,1,1,1,1,3,2,2,3,...,2,3,0,3,0,0,2,2,2,3
fires,0,0,0,1,2,3,0,0,0,0,...,3,2,4,4,6,8,4,4,2,2
life_frames,20,30,30,20,20,20,25,25,25,30,...,30,35,30,30,30,30,30,30,30,30


## 4. Experiment settings
All notebooks read these settings so every agent is tested on the same (level, seed) worlds.
50 seeds x 24 levels = 1200 episodes per agent.

In [16]:
CFG = {
    'levels': list(range(1, 25)),
    'n_seeds': 50,                          # random layouts per level (paired across agents)
    'time_limit_s': 60.0,                   # an episode not finished by then counts as a failure
    'stress_speeds': [1.0, 1.25, 1.5, 2.0], # hazard speed multipliers for the robustness test
    'stress_seeds': 20,
    'hard_levels': [7, 10, 11, 15, 16, 18, 21, 22, 23, 24],   # used for the sensitivity study
    'sensitivity_seeds': 20,
}
json.dump(CFG, open(f"{PATHS['config']}/experiment_config.json", 'w'), indent=2)

import platform, numpy, pygad, matplotlib
env = {'python': platform.python_version(), 'numpy': numpy.__version__, 'pygad': pygad.__version__,
       'matplotlib': matplotlib.__version__, 'cpus': os.cpu_count(),
       'cointex_parta': cointex_parta.__version__}
json.dump(env, open(f"{PATHS['config']}/environment.json", 'w'), indent=2)
log_run('00_setup', **env)
CFG, env

({'levels': [1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24],
  'n_seeds': 50,
  'time_limit_s': 60.0,
  'stress_speeds': [1.0, 1.25, 1.5, 2.0],
  'stress_seeds': 20,
  'hard_levels': [7, 10, 11, 15, 16, 18, 21, 22, 23, 24],
  'sensitivity_seeds': 20},
 {'python': '3.13.15',
  'numpy': '2.1.3',
  'pygad': '3.3.1',
  'matplotlib': '3.10.0',
  'cpus': 12,
  'cointex_parta': '1.0.0'})

## 5. Quick run - one episode per agent (level 23, seed 0)

In [17]:
from cointex_parta import runner
for agent in ['greedy', 'ga_task2', 'proposed']:
    r = runner.play_episode(23, 0, agent)
    print(f"{agent:10s} won={r['won']} coins={r['coins_collected']}/{r['coins_total']} "
          f"time={r['seconds']:.1f}s life={r['remaining_life']:.0f}% plan={r['mean_plan_ms']:.1f}ms")

greedy     won=0 coins=8/25 time=3.3s life=0% plan=0.0ms
ga_task2   won=1 coins=25/25 time=10.2s life=63% plan=8.9ms
proposed   won=1 coins=25/25 time=7.5s life=100% plan=5.4ms


Setup done. The experiments are in notebooks 01 to 04.